## --- Section: 01_attention_mechanism ---

# The Transformer Architecture


In [ ]:
import torch
import torch.nn as nn
import math


In [ ]:
# 1. Self-Attention Mechanism (Conceptual)
# Unlike RNNs, Transformers process all words at the SAME TIME.
# Attention calculates how much 'focus' a word should have on every other word in the sentence.
attention = nn.MultiheadAttention(embed_dim=256, num_heads=8)


In [ ]:
# 2. Positional Encoding
# Since words are processed at the same time, the model loses the order of the sentence.
# Positional encoding injects 'time' back into the words using sine and cosine waves.


## --- Section: 01_transformers_basics ---

# Transformers Architecture

Introduced in the famous 2017 paper *"Attention Is All You Need"*, the Transformer architecture revolutionized Deep Learning, replacing LSTMs and paving the way for Large Language Models (LLMs) like GPT and BERT.

## The Problems with LSTMs
1. **Sequential Bottleneck:** LSTMs must read a sentence word-by-word. You cannot parallelize training, making them incredibly slow to train on massive datasets.
2. **Information Loss:** Even LSTMs struggle to remember the start of a 1000-word paragraph by the time they reach the end.

## Core Concepts of Transformers
1. **Self-Attention:** Instead of reading word-by-word, a Transformer looks at the *entire* sentence at once. For every word, it calculates an "attention score" against every other word in the sentence to understand the context (e.g., figuring out that "bank" refers to a river bank, not a financial bank, based on surrounding words).
2. **Positional Encoding:** Because the Transformer processes all words simultaneously (solving the sequential bottleneck), it doesn't inherently know word order. Positional encoding adds a mathematical timestamp to every word so the network knows its position.
3. **Encoder-Decoder:** The original Transformer had an Encoder (to understand input) and a Decoder (to generate output). Models like BERT use only the Encoder; models like GPT use only the Decoder.

In [ ]:
import torch.nn as nn

class TransformerEncoderBlock(nn.Module):
    """A single block of a Transformer Encoder (like those used in BERT)."""
    def __init__(self, embed_size, heads):
        super(TransformerEncoderBlock, self).__init__()
        # Multi-Head Attention allows the model to focus on different parts of the sentence simultaneously
        self.attention = nn.MultiheadAttention(embed_dim=embed_size, num_heads=heads, batch_first=True)
        
        # Standard Feed Forward Network
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size)
        )
        
        # Layer Normalization stabilizes training
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        
    def forward(self, x):
        # 1. Self-Attention with a Residual Connection
        attention_out, _ = self.attention(query=x, key=x, value=x)
        x = self.norm1(attention_out + x)
        
        # 2. Feed Forward with a Residual Connection
        forward_out = self.feed_forward(x)
        out = self.norm2(forward_out + x)
        return out